# Module 3.6 — Unsteady Flows and Time-Stepping

**The distinction that matters in practice:**
Module 2 used time-stepping to drive toward a steady state. The time step was chosen for stability (large as possible), not accuracy — because transient behaviour was irrelevant. For truly unsteady flows, the time step must be chosen for **accuracy**, and the scheme must be at least second-order in time.

**When a flow is genuinely unsteady:**
- Vortex shedding (Kármán street, cylinder flow)
- Turbomachinery (rotor-stator interaction, blade passing)
- Cardiovascular flows (pulsatile blood flow)
- Combustion instabilities and acoustic resonance
- Weather and ocean circulation (inherently transient)

**Roadmap:**
1. Explicit vs. implicit time integration — the accuracy vs. stability trade-off
2. Order of accuracy in time — what it means and why it matters
3. Stability regions — visualising when schemes blow up
4. Runge-Kutta methods — 4th order with only 4 function evaluations
5. Dual time-stepping — the practical approach for unsteady incompressible flows
6. Choosing $\Delta t$ — CFL for accuracy, not just stability

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Time Integration Schemes

### The model ODE

All analysis starts with the test equation $du/dt = \lambda u$ (complex $\lambda$). For CFD:
- Advection eigenvalues: $\lambda = -ic k\Delta x / \Delta t$ — purely imaginary
- Diffusion eigenvalues: $\lambda = -4r\sin^2(k\Delta x/2)$ — negative real

A scheme is stable if the amplification factor $|G| = |u^{n+1}/u^n| \leq 1$ for all relevant $\lambda\Delta t$.

### Common schemes and their properties

**Forward Euler (explicit, 1st order):**
$$u^{n+1} = u^n + \Delta t\,F(u^n)$$
Simple, cheap. Stability: only a small region around the origin in the $\lambda\Delta t$ plane. Requires very small $\Delta t$ for stiff problems.

**Adams-Bashforth 2 (explicit, 2nd order):**
$$u^{n+1} = u^n + \Delta t\left(\frac{3}{2}F(u^n) - \frac{1}{2}F(u^{n-1})\right)$$
Uses two previous time levels. Better accuracy, similar stability region to Euler.

**Crank-Nicolson (implicit, 2nd order, A-stable for diffusion):**
$$u^{n+1} = u^n + \frac{\Delta t}{2}\left[F(u^n) + F(u^{n+1})\right]$$
Solves a nonlinear system each step (or linear if $F$ is linear). Unconditionally stable for diffusion. Can oscillate for large $\Delta t$ with pure advection.

**BDF2 (implicit, 2nd order):**
$$\frac{3u^{n+1} - 4u^n + u^{n-1}}{2\Delta t} = F(u^{n+1})$$
Used in OpenFOAM as `backward`. More damping than CN — preferred when stability is critical.

**RK4 (explicit, 4th order):**
$$k_1 = F(u^n), \quad k_2 = F(u^n+\tfrac{\Delta t}{2}k_1), \quad k_3 = F(u^n+\tfrac{\Delta t}{2}k_2), \quad k_4 = F(u^n+\Delta t k_3)$$
$$u^{n+1} = u^n + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$
Four function evaluations per step but fourth-order accuracy. Stability region covers part of the imaginary axis — useful for wave propagation problems.

### Choosing $\Delta t$ for accuracy

For steady-state: choose the largest stable $\Delta t$.

For time-accurate unsteady: choose $\Delta t$ based on the **physical time scale** you need to resolve. If the Kármán shedding frequency is $f = 2$ Hz and you want 100 points per cycle: $\Delta t = 1/(100 \times 2) = 0.005$ s. Then check CFL is also satisfied.

In [ ]:
# ── Compare time integration schemes on du/dt = -u (exact: u = exp(-t)) ───────

def euler_fwd(u0, dt, T):
    u, t, us, ts = u0, 0.0, [u0], [0.0]
    while t < T:
        u += dt*(-u); t += dt
        us.append(u); ts.append(t)
    return np.array(ts), np.array(us)

def adams_bashforth2(u0, dt, T):
    u, t = u0, 0.0
    F_old = -u0
    u += dt*F_old; t += dt   # Euler first step
    us, ts = [u0, u], [0.0, t]
    while t < T:
        F_new = -u
        u = u + dt*(1.5*F_new - 0.5*F_old)
        F_old = F_new; t += dt
        us.append(u); ts.append(t)
    return np.array(ts), np.array(us)

def rk4(u0, dt, T):
    u, t, us, ts = u0, 0.0, [u0], [0.0]
    while t < T:
        k1 = -u
        k2 = -(u + dt/2*k1)
        k3 = -(u + dt/2*k2)
        k4 = -(u + dt*k3)
        u += dt/6*(k1 + 2*k2 + 2*k3 + k4)
        t += dt; us.append(u); ts.append(t)
    return np.array(ts), np.array(us)

T   = 5.0
dt  = 0.5   # large time step to show accuracy differences
t_ex = np.linspace(0, T, 300)
u_ex = np.exp(-t_ex)

t_eu, u_eu = euler_fwd(1.0, dt, T)
t_ab, u_ab = adams_bashforth2(1.0, dt, T)
t_rk, u_rk = rk4(1.0, dt, T)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Trajectory
axes[0].plot(t_ex, u_ex,  'k-', lw=2.5, label='Exact: $e^{-t}$')
axes[0].plot(t_eu, u_eu, 'r-o', ms=5, lw=1.5, label='Euler (1st order)')
axes[0].plot(t_ab, u_ab, 'b-s', ms=5, lw=1.5, label='Adams-Bashforth2 (2nd)')
axes[0].plot(t_rk, u_rk, 'g-^', ms=5, lw=1.5, label='RK4 (4th order)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('u')
axes[0].set_title(f'$du/dt = -u$, $\\Delta t = {dt}$ (large step to show error)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Convergence
dt_vals = [0.5, 0.25, 0.125, 0.0625, 0.03125]
u_ref   = np.exp(-T)
errs_eu = [abs(euler_fwd(1.0, dt, T)[1][-1] - u_ref)         for dt in dt_vals]
errs_ab = [abs(adams_bashforth2(1.0, dt, T)[1][-1] - u_ref)  for dt in dt_vals]
errs_rk = [abs(rk4(1.0, dt, T)[1][-1] - u_ref)               for dt in dt_vals]

dts = np.array(dt_vals)
axes[1].loglog(dts, errs_eu, 'r-o', ms=6, lw=2, label='Euler')
axes[1].loglog(dts, errs_ab, 'b-s', ms=6, lw=2, label='AB2')
axes[1].loglog(dts, errs_rk, 'g-^', ms=6, lw=2, label='RK4')
axes[1].loglog(dts, 0.5*dts**1,  'k--', alpha=0.5, label='$O(\\Delta t)$')
axes[1].loglog(dts, 0.2*dts**2,  'k-',  alpha=0.5, label='$O(\\Delta t^2)$')
axes[1].loglog(dts, 0.02*dts**4, 'k:',  alpha=0.5, label='$O(\\Delta t^4)$')
axes[1].set_xlabel('$\\Delta t$'); axes[1].set_ylabel('Error at $t=5$')
axes[1].set_title('Temporal order of accuracy\nSteeper slope = higher order')
axes[1].legend(fontsize=8); axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print('For the same Δt:')
print(f'  Euler error: {errs_eu[0]:.4f}')
print(f'  AB2 error:   {errs_ab[0]:.4f}   ({errs_eu[0]/errs_ab[0]:.1f}× smaller)')
print(f'  RK4 error:   {errs_rk[0]:.6f}   ({errs_eu[0]/errs_rk[0]:.0f}× smaller)')

## 2. Stability Regions — Visualising When Schemes Blow Up

### The amplification factor

For $du/dt = \lambda u$ with complex $\lambda$, define $z = \lambda\Delta t$. Then:

- **Forward Euler:** $G = 1 + z$
- **RK4:** $G = 1 + z + z^2/2 + z^3/6 + z^4/24$

Stability: $|G(z)| \leq 1$.

**Physical interpretation of $z$:**
- For diffusion: $z = -r$ (real, negative) — must have $|z| \leq 2$ for Euler
- For advection: $z = -i\nu\cdot$(some factor) — imaginary — RK4 covers part of imaginary axis but Euler does not

**Consequence for CFD:**
- Euler is unstable for pure advection (imaginary axis not inside stability region)
- RK4 is stable for mild advection (imaginary axis enters stability region)
- Implicit schemes (CN, BDF2) have the entire left half-plane inside — unconditionally stable for diffusion

## 3. Dual Time-Stepping for Unsteady Incompressible Flow

SIMPLE was designed for steady state. For unsteady flows, you want accurate time integration. The challenge: each time step requires multiple pressure-velocity iterations to converge. **Dual time-stepping** solves this:

```
OUTER LOOP (physical time t): small Δt for accuracy
  For each physical time step:
    INNER LOOP (pseudo-time τ): large Δτ for fast convergence
      Apply SIMPLE with large Δτ
      Repeat until inner residuals < tolerance
    → Solution at physical time t^{n+1} converged
```

The physical time derivative becomes a source term in the pseudo-time equation:

$$\frac{\partial\mathbf{u}}{\partial\tau} + \frac{3\mathbf{u}^{n+1} - 4\mathbf{u}^n + \mathbf{u}^{n-1}}{2\Delta t} + N(\mathbf{u}^{n+1}) = 0$$

This is BDF2 in physical time (2nd order accurate) plus pseudo-time advancement to converge the system.

**OpenFOAM:** `pimpleFoam` uses PIMPLE = PISO + SIMPLE with dual time-stepping.

## Summary

| Scheme | Order | Stability | Cost | When to use |
|--------|-------|-----------|------|-------------|
| Forward Euler | 1st | Limited | $O(N)$ | Only steady-state pseudo-time |
| Adams-Bashforth 2 | 2nd | Limited | $O(N)$ | Unsteady explicit, moderate accuracy |
| Crank-Nicolson | 2nd | A-stable (diffusion) | $O(N)$ + solve | High accuracy, smooth flows |
| BDF2 | 2nd | More damping | $O(N)$ + solve | Unsteady, stability preferred |
| RK4 | 4th | Wide region | $4\times O(N)$ | Wave propagation, spectral problems |

**Choosing $\Delta t$:** For unsteady flows always use $Co < 1$ (preferably $Co < 0.5$) AND ensure the physical time scales of interest are resolved with $\geq 50$–100 steps per period.

---
**Next:** Module 3.7 — Compressible Flow: Euler equations, Mach number, shock capturing, Sod tube benchmark.